# S2 — ML solar energy estimation across Brazilian climatic regions (NASA/POWER)
**Pipeline revisado para o artigo S2 (CNPq 401718/2023-9).**

O que este notebook faz, de ponta a ponta:
1. Baixa os dados diários do NASA/POWER **direto da API** (não precisa subir CSV) para 10 cidades, 2020–2025.
2. Treina 6 modelos (Linear, CART, SVM, RF, GBT, GAM) em **2020–2023** e testa em **2024** e **2025** (validação temporal estrita).
3. Compara **três conjuntos de preditores** (o ganho entre eles é parte do resultado do artigo):
   - **F0 – Meteorológico**: T2M, RH2M, WS2M, PS, PRECTOTCORR (= rascunho atual)
   - **F1 – Meteo + astronômico**: F0 + radiação no topo da atmosfera, dia do ano (sen/cos), amplitude térmica diária
   - **F2 – F1 + nebulosidade**: F1 + CLOUD_AMT (derivado de satélite; reportar como cenário separado)
4. Compara com **referências ingênuas** (climatologia e persistência) → *skill score*.
5. Teste de **Diebold–Mariano** (melhor modelo vs climatologia).
6. Gera tabelas (CSV/XLSX, com ponto decimal correto) e figuras em inglês, 300 dpi, e empacota tudo em `S2_outputs.zip`.

**Como usar:** `Ambiente de execução → Executar tudo`. Leva ~10–20 min. No final, baixe `S2_outputs.zip`.

In [ ]:
# !pip -q install requests openpyxl   # (já vêm no Colab; descomente se faltar)
import os, io, json, time, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})

OUT = "S2_outputs"
os.makedirs(f"{OUT}/figures", exist_ok=True)
os.makedirs(f"{OUT}/tables", exist_ok=True)
os.makedirs(f"{OUT}/raw_data", exist_ok=True)

SEED = 42
TRAIN_YEARS = [2020, 2021, 2022, 2023]
TEST_YEARS = [2024, 2025]
START, END = "20200101", "20251231"

# Cidades (lat, lon) e região climática
CITIES = {
    "Manaus":         (-3.119,  -60.022, "North"),
    "Fortaleza":      (-3.732,  -38.527, "Northeast"),
    "Recife":         (-8.054,  -34.881, "Northeast"),
    "Brasilia":       (-15.794, -47.882, "Center-West"),
    "Cuiaba":         (-15.601, -56.097, "Center-West"),
    "Belo Horizonte": (-19.917, -43.935, "Southeast"),
    "Rio de Janeiro": (-22.907, -43.173, "Southeast"),
    "Sao Paulo":      (-23.550, -46.633, "Southeast"),
    "Curitiba":       (-25.428, -49.273, "South"),
    "Porto Alegre":   (-30.035, -51.218, "South"),
}

PARAMS = ["ALLSKY_SFC_SW_DWN", "TOA_SW_DWN", "T2M", "T2M_MAX", "T2M_MIN",
          "RH2M", "WS2M", "PS", "PRECTOTCORR", "CLOUD_AMT"]
TARGET = "ALLSKY_SFC_SW_DWN"   # kWh/m²/day (community=RE)

## 1. Download NASA/POWER (API diária, comunidade RE)

In [ ]:
def fetch_power(lat, lon, start=START, end=END, params=PARAMS, retries=4):
    import requests
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    q = {"parameters": ",".join(params), "community": "RE", "longitude": lon,
         "latitude": lat, "start": start, "end": end, "format": "JSON",
         "time-standard": "LST"}
    for k in range(retries):
        try:
            r = requests.get(url, params=q, timeout=120)
            r.raise_for_status()
            p = r.json()["properties"]["parameter"]
            df = pd.DataFrame(p)
            df.index = pd.to_datetime(df.index, format="%Y%m%d")
            df.index.name = "date"
            return df.replace(-999.0, np.nan)
        except Exception as e:
            print(f"   tentativa {k+1} falhou: {e}")
            time.sleep(5 * (k + 1))
    raise RuntimeError("Falha no download NASA/POWER")


def load_all(cities=CITIES, fetch=fetch_power):
    frames = {}
    for city, (lat, lon, reg) in cities.items():
        path = f"{OUT}/raw_data/{city.replace(' ', '_')}.csv"
        if os.path.exists(path):
            df = pd.read_csv(path, index_col=0, parse_dates=True)
        else:
            print(f"Baixando {city}...")
            df = fetch(lat, lon)
            df.to_csv(path)
        frames[city] = df
    return frames

## 2. Engenharia de atributos e conjuntos de preditores

In [ ]:
F0 = ["T2M", "RH2M", "WS2M", "PS", "PRECTOTCORR"]
F1 = F0 + ["TOA_SW_DWN", "DTR", "DOY_SIN", "DOY_COS"]
F2 = F1 + ["CLOUD_AMT"]
FEATURE_SETS = {"F0_meteo": F0, "F1_meteo_astro": F1, "F2_meteo_astro_cloud": F2}


def build_features(df):
    d = df.copy()
    d["DTR"] = d["T2M_MAX"] - d["T2M_MIN"]          # amplitude térmica (proxy de nebulosidade)
    doy = d.index.dayofyear
    d["DOY_SIN"] = np.sin(2 * np.pi * doy / 365.25)
    d["DOY_COS"] = np.cos(2 * np.pi * doy / 365.25)
    d["YEAR"] = d.index.year
    return d

## 3. Modelos, referências e métricas

In [ ]:
def make_models():
    """6 modelos do artigo. Hiperparâmetros ajustados por GridSearch com TimeSeriesSplit
    apenas nos anos de treino (sem vazamento)."""
    return {
        "Linear": (Pipeline([("sc", StandardScaler()), ("m", LinearRegression())]), None),
        "CART": (DecisionTreeRegressor(random_state=SEED),
                 {"max_depth": [4, 6, 8, 12], "min_samples_leaf": [5, 10, 20]}),
        "SVM": (Pipeline([("sc", StandardScaler()), ("m", SVR())]),
                {"m__C": [1, 10, 30], "m__gamma": ["scale", 0.05], "m__epsilon": [0.1, 0.3]}),
        "RF": (RandomForestRegressor(n_estimators=400, random_state=SEED, n_jobs=-1),
               {"max_depth": [None, 12], "min_samples_leaf": [1, 5], "max_features": [0.5, 1.0]}),
        "GBT": (GradientBoostingRegressor(random_state=SEED),
                {"n_estimators": [200, 500], "learning_rate": [0.03, 0.1], "max_depth": [2, 3]}),
        # GAM implementado como modelo aditivo de splines penalizadas (equivalente a GAM gaussiano)
        "GAM": (Pipeline([("sc", StandardScaler()),
                          ("sp", SplineTransformer(n_knots=8, degree=3)),
                          ("m", Ridge())]),
                {"m__alpha": [0.1, 1, 10]}),
    }


def metrics(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    rmse = np.sqrt(mean_squared_error(y, yhat))
    return {"MAE": mean_absolute_error(y, yhat), "RMSE": rmse,
            "nRMSE_%": 100 * rmse / np.mean(y), "MBE": np.mean(yhat - y),
            "R2": r2_score(y, yhat)}


def climatology_ref(train, test):
    """Média por dia do ano nos anos de treino, suavizada (janela 15 d, circular)."""
    clim = train.groupby(train.index.dayofyear)[TARGET].mean().reindex(range(1, 367))
    clim = clim.interpolate(limit_direction="both")
    ext = pd.concat([clim.iloc[-7:], clim, clim.iloc[:7]])
    clim = ext.rolling(15, center=True).mean().iloc[7:-7]
    return clim.reindex(test.index.dayofyear).values


def persistence_ref(full, test):
    return full[TARGET].shift(1).reindex(test.index).values


def diebold_mariano(e1, e2, h=1):
    """DM com correção de Harvey-Leybourne-Newbold, perda quadrática.
    DM<0 → modelo 1 melhor. Retorna (estatística, p-valor bicaudal)."""
    d = np.asarray(e1) ** 2 - np.asarray(e2) ** 2
    d = d[~np.isnan(d)]
    n = len(d)
    dbar = d.mean()
    gamma = [np.sum((d[k:] - dbar) * (d[:n - k] - dbar)) / n for k in range(h)]
    var = (gamma[0] + 2 * sum(gamma[1:])) / n
    dm = dbar / np.sqrt(var)
    hln = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    stat = dm * hln
    p = 2 * stats.t.sf(np.abs(stat), df=n - 1)
    return stat, p

## 4. Execução principal

In [ ]:
def run(frames, feature_sets=FEATURE_SETS, test_years=TEST_YEARS, tune=True, verbose=True):
    rows, dm_rows, preds, fitted = [], [], {}, {}
    for city, raw in frames.items():
        region = CITIES[city][2]
        d = build_features(raw)
        for fs_name, cols in feature_sets.items():
            dd = d.dropna(subset=cols + [TARGET])
            train = dd[dd.YEAR.isin(TRAIN_YEARS)]
            Xtr, ytr = train[cols], train[TARGET]
            tscv = TimeSeriesSplit(n_splits=4)
            for mname, (est, grid) in make_models().items():
                if tune and grid:
                    gs = GridSearchCV(est, grid, cv=tscv, scoring="neg_root_mean_squared_error", n_jobs=-1)
                    gs.fit(Xtr, ytr)
                    model, best = gs.best_estimator_, gs.best_params_
                else:
                    model, best = est.fit(Xtr, ytr), {}
                fitted[(city, fs_name, mname)] = (model, best)
                for ty in test_years:
                    test = dd[dd.YEAR == ty]
                    if len(test) < 30:
                        continue
                    yhat = model.predict(test[cols])
                    clim = climatology_ref(train, test)
                    pers = persistence_ref(dd, test)
                    m = metrics(test[TARGET], yhat)
                    rmse_c = np.sqrt(np.nanmean((test[TARGET].values - clim) ** 2))
                    mask = ~np.isnan(pers)
                    rmse_p = np.sqrt(np.mean((test[TARGET].values[mask] - pers[mask]) ** 2))
                    m.update({"Skill_vs_Climatology": 1 - m["RMSE"] / rmse_c,
                              "Skill_vs_Persistence": 1 - m["RMSE"] / rmse_p,
                              "RMSE_Climatology": rmse_c, "RMSE_Persistence": rmse_p})
                    rows.append({"City": city, "Region": region, "FeatureSet": fs_name,
                                 "Model": mname, "TestYear": ty, "n_test": len(test),
                                 **m, "BestParams": json.dumps(best)})
                    preds[(city, fs_name, mname, ty)] = pd.Series(yhat, index=test.index)
                    preds[(city, fs_name, "_obs", ty)] = test[TARGET]
                    preds[(city, fs_name, "_clim", ty)] = pd.Series(clim, index=test.index)
            if verbose:
                print(f"OK {city:15s} {fs_name}")
    res = pd.DataFrame(rows)
    # Diebold-Mariano: melhor modelo (menor RMSE) vs climatologia, por cidade/conjunto/ano
    for (city, fs, ty), g in res.groupby(["City", "FeatureSet", "TestYear"]):
        best = g.loc[g.RMSE.idxmin(), "Model"]
        obs = preds[(city, fs, "_obs", ty)]
        e_m = obs - preds[(city, fs, best, ty)]
        e_c = obs - preds[(city, fs, "_clim", ty)]
        s, p = diebold_mariano(e_m.values, e_c.values)
        dm_rows.append({"City": city, "FeatureSet": fs, "TestYear": ty, "BestModel": best,
                        "DM_stat": s, "p_value": p, "Best_beats_climatology_5%": (s < 0) and (p < 0.05)})
    return res, pd.DataFrame(dm_rows), preds, fitted

## 5. Tabelas e figuras

In [ ]:
MODEL_ORDER = ["Linear", "CART", "SVM", "RF", "GBT", "GAM"]
CITY_ORDER = list(CITIES.keys())


def save_tables(res, dm):
    res.to_csv(f"{OUT}/tables/results_all.csv", index=False, float_format="%.4f")
    dm.to_csv(f"{OUT}/tables/diebold_mariano.csv", index=False, float_format="%.4f")
    with pd.ExcelWriter(f"{OUT}/tables/S2_results.xlsx") as xw:
        res.to_excel(xw, sheet_name="all_results", index=False)
        dm.to_excel(xw, sheet_name="diebold_mariano", index=False)
        for fs in res.FeatureSet.unique():
            t = res[(res.FeatureSet == fs) & (res.TestYear == 2024)]
            for met in ["MAE", "RMSE", "R2", "Skill_vs_Climatology"]:
                pv = t.pivot(index="City", columns="Model", values=met).reindex(index=CITY_ORDER, columns=MODEL_ORDER)
                pv.round(3).to_excel(xw, sheet_name=f"{fs[:2]}_{met[:14]}_2024")
        # Tabela-síntese do artigo: ganho por conjunto de preditores (melhor modelo por cidade, 2024)
        best = (res[res.TestYear == 2024].sort_values("RMSE").groupby(["City", "FeatureSet"]).first()
                .reset_index()[["City", "Region", "FeatureSet", "Model", "MAE", "RMSE", "nRMSE_%", "R2", "Skill_vs_Climatology"]])
        best.round(3).to_excel(xw, sheet_name="best_per_city_2024", index=False)
    return best


def fig_r2_heatmaps(res, year=2024):
    fss = list(FEATURE_SETS)
    fig, axes = plt.subplots(1, len(fss), figsize=(4.2 * len(fss), 4.4), sharey=True)
    for ax, fs in zip(np.atleast_1d(axes), fss):
        pv = res[(res.FeatureSet == fs) & (res.TestYear == year)].pivot(
            index="City", columns="Model", values="R2").reindex(index=CITY_ORDER, columns=MODEL_ORDER)
        im = ax.imshow(pv.values, cmap="RdBu", vmin=-1, vmax=1, aspect="auto")
        ax.set_xticks(range(len(MODEL_ORDER))); ax.set_xticklabels(MODEL_ORDER, rotation=45, ha="right")
        ax.set_yticks(range(len(CITY_ORDER))); ax.set_yticklabels(CITY_ORDER)
        for i in range(pv.shape[0]):
            for j in range(pv.shape[1]):
                v = pv.values[i, j]
                if np.isnan(v):
                    continue
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                        color="white" if abs(v) > 0.6 else "black")
        ax.set_title(fs.replace("_", " "))
    fig.colorbar(im, ax=axes, shrink=0.8, label="R² (test year %d)" % year)
    fig.savefig(f"{OUT}/figures/Fig_R2_heatmap_{year}.png", bbox_inches="tight")
    plt.close(fig)


def fig_skill(res, year=2024):
    t = res[res.TestYear == year].sort_values("RMSE").groupby(["City", "FeatureSet"]).first().reset_index()
    pv = t.pivot(index="City", columns="FeatureSet", values="Skill_vs_Climatology").reindex(CITY_ORDER)
    ax = pv.plot(kind="bar", figsize=(8, 3.8), width=0.8, color=["#9aa5b1", "#3b7dd8", "#1b3f73"])
    ax.axhline(0, color="k", lw=0.8)
    ax.set_ylabel("Skill score vs climatology (best model)")
    ax.set_xlabel(""); ax.legend(title="Predictor set", fontsize=8)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout(); plt.savefig(f"{OUT}/figures/Fig_skill_vs_climatology_{year}.png"); plt.close()


def fig_timeseries(res, preds, fs="F1_meteo_astro", year=2024):
    fig, axes = plt.subplots(5, 2, figsize=(10, 11), sharex=True)
    for ax, city in zip(axes.ravel(), CITY_ORDER):
        g = res[(res.City == city) & (res.FeatureSet == fs) & (res.TestYear == year)]
        if g.empty:
            ax.axis("off"); continue
        best = g.loc[g.RMSE.idxmin(), "Model"]
        obs = preds[(city, fs, "_obs", year)]
        ax.plot(obs.index, obs.values, color="k", lw=0.8, label="NASA/POWER")
        ax.plot(obs.index, preds[(city, fs, best, year)].values, color="#d8573b", lw=0.8, label=best)
        ax.plot(obs.index, preds[(city, fs, "_clim", year)].values, color="#3b7dd8", lw=0.8, ls="--", label="Climatology")
        r2 = g.loc[g.RMSE.idxmin(), "R2"]
        ax.set_title(f"{city} — {best} (R² = {r2:.2f})", fontsize=9)
        ax.set_ylabel("kWh m$^{-2}$ day$^{-1}$")
        ax.legend(fontsize=6, loc="lower left", ncol=3)
    fig.autofmt_xdate(); fig.tight_layout()
    fig.savefig(f"{OUT}/figures/Fig_timeseries_{fs}_{year}.png"); plt.close(fig)


def fig_corr(frames):
    """Correlação de Spearman entre cada preditor e a energia solar diária, por cidade (corrige a Fig. 4)."""
    cols = ["T2M", "DTR", "RH2M", "WS2M", "PS", "PRECTOTCORR", "TOA_SW_DWN", "CLOUD_AMT"]
    M = pd.DataFrame({c: [build_features(frames[c])[[k, TARGET]].corr("spearman").iloc[0, 1] for k in cols]
                      for c in CITY_ORDER if c in frames}, index=cols).T
    fig, ax = plt.subplots(figsize=(6.5, 4.6))
    im = ax.imshow(M.values, cmap="RdBu", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=45, ha="right")
    ax.set_yticks(range(len(M))); ax.set_yticklabels(M.index)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f"{M.values[i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if abs(M.values[i, j]) > 0.6 else "black")
    fig.colorbar(im, label="Spearman ρ with daily solar energy")
    fig.tight_layout(); fig.savefig(f"{OUT}/figures/Fig_correlation_predictors_vs_solar.png"); plt.close(fig)
    M.round(3).to_csv(f"{OUT}/tables/correlation_predictors_vs_solar.csv")


def fig_importance(frames, fitted, fs="F1_meteo_astro"):
    cols = FEATURE_SETS[fs]
    imp = {}
    for city in CITY_ORDER:
        if (city, fs, "RF") not in fitted:
            continue
        d = build_features(frames[city]).dropna(subset=cols + [TARGET])
        test = d[d.YEAR == 2024]
        pi = permutation_importance(fitted[(city, fs, "RF")][0], test[cols], test[TARGET],
                                    n_repeats=10, random_state=SEED, scoring="neg_root_mean_squared_error")
        imp[city] = pi.importances_mean
    M = pd.DataFrame(imp, index=cols).T
    M.round(4).to_csv(f"{OUT}/tables/permutation_importance_RF_{fs}.csv")
    ax = (M.div(M.sum(axis=1), axis=0)).plot(kind="barh", stacked=True, figsize=(7.5, 4.5), colormap="tab20")
    ax.set_xlabel("Relative permutation importance (RF, test 2024)")
    ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout(); plt.savefig(f"{OUT}/figures/Fig_importance_RF_{fs}.png"); plt.close()

## 6. Rodar tudo

In [ ]:
if __name__ == "__main__" or "get_ipython" in globals():
    if not os.environ.get("S2_SKIP_MAIN"):
        frames = load_all()
        res, dm, preds, fitted = run(frames)
        best = save_tables(res, dm)
        fig_r2_heatmaps(res); fig_skill(res); fig_corr(frames)
        for fs in FEATURE_SETS:
            fig_timeseries(res, preds, fs=fs)
        fig_importance(frames, fitted)

        print("\n=== Melhor modelo por cidade e conjunto de preditores (teste 2024) ===")
        print(best.round(3).to_string(index=False))
        print("\n=== Diebold–Mariano (melhor modelo vs climatologia) ===")
        print(dm.round(3).to_string(index=False))

        with zipfile.ZipFile("S2_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
            for root, _, files in os.walk(OUT):
                for f in files:
                    z.write(os.path.join(root, f))
        print("\nArquivo gerado: S2_outputs.zip")
        try:
            from google.colab import files
            files.download("S2_outputs.zip")
        except Exception:
            pass